## **Scrolling Through Change Final Project**
Maya Patel & Vyshnavi Telukuntla

### 1) Baseline Model

In [ ]:
import random
import numpy as np
import torch
import pandas as pd
import re
from torch.utils.data import DataLoader

random.seed(70)
np.random.seed(70)
torch.manual_seed(70)

In [ ]:
df = pd.read_csv("caption_final.csv")

# Check for missing values
# df.isna().any()
# df.isna().sum()

# Preprocess captions (lowercase, remove URLs, normalize spacing)
def clean_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"\s+", " ", text)
    return text

df['clean_caption'] = df['caption'].astype(str).apply(clean_text)

print(df)

# Explore dataset
print("Dataset shape:", df.shape)
print("Number of unique accounts:", df['username'].nunique())

# New feature: number of words in each caption
df['caption_length'] = df['clean_caption'].apply(lambda x: len(x.split()))
print("Average caption length:", df['caption_length'].mean())

if 'account_type' in df.columns:
  print(df['account_type'].value_counts())
  print(df['account_type'].unique())

In [ ]:
# Train/Validation/Test Split
from sklearn.model_selection import train_test_split

accounts = df['username'].unique()

# Split accounts into train (80%) and test(20%)
train_acc, test_acc = train_test_split(accounts, test_size = 0.2, random_state = 50)

# Split remaining 80% into train (70%) and validation (10%)
train_acc, val_acc = train_test_split(train_acc, test_size = 0.125, random_state = 50)

# Create datasets based on account username
train_df = df[df['username'].isin(train_acc)]
val_df = df[df['username'].isin(val_acc)]
test_df = df[df['username'].isin(test_acc)]

# Class balance check
print("Label counts:")  # Overall distribution of labels
print(df['label'].value_counts())

# Check label distribution in each split
print("\nTrain label counts:")
print(train_df['label'].value_counts())

print("\nValidation label counts:")
print(val_df['label'].value_counts())

print("\nTest label counts:")
print(test_df['label'].value_counts())

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, recall_score, precision_score, accuracy_score

# Baseline model
vectorizer = TfidfVectorizer(max_features = 5000)

X_train = vectorizer.fit_transform(train_df['clean_caption'])
X_val = vectorizer.transform(val_df['clean_caption'])
X_test = vectorizer.transform(test_df['clean_caption'])

y_train = train_df['label']
y_val = val_df['label']
y_test = test_df['label']

logreg_model = LogisticRegression(max_iter = 1000)
logreg_model.fit(X_train, y_train)

feature_names = vectorizer.get_feature_names_out()
coeffs = logreg_model.coef_[0]

# Validation evaluation
y_val_pred = logreg_model.predict(X_val)
print("Validation F1:", f1_score(y_val, y_val_pred, average = 'macro'))

# Test evaluation
y_pred = logreg_model.predict(X_test)

# Breakdown of performance per class
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Evaluation metrics
print("F1 Score:", f1_score(y_test, y_pred, average = 'macro'))
print("Recall:", recall_score(y_test, y_pred, average = 'macro'))
print("Precision:", precision_score(y_test, y_pred, average = 'macro'))
print("Accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
# Confusion matrix
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize = (8, 6))
sns.heatmap(cm, annot = True, fmt = 'd', cmap = 'Blues', cbar = False)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title("Baseline Confusion Matrix")
plt.show()

In [ ]:
# Group comparison
inf_test = test_df[test_df['account_type'] == 'influencer'] # Split test set
pol_test = test_df[test_df['account_type'] == 'politician']

print("Influencer test examples:", len(inf_test))
print("Politician test examples:", len(pol_test))

X_inf = vectorizer.transform(inf_test['clean_caption'])
y_inf = inf_test['label']

X_pol = vectorizer.transform(pol_test['clean_caption'])
y_pol = pol_test['label']

# Predict for each group
y_inf_pred = logreg_model.predict(X_inf)
y_pol_pred = logreg_model.predict(X_pol)

# Influencer performance
print("Influencer F1 Score:", f1_score(y_inf, y_inf_pred, average = 'macro'))
print("Influencer Accuracy:", accuracy_score(y_inf, y_inf_pred))

# Politician performance
print("Politician F1 Score:", f1_score(y_pol, y_pol_pred, average = 'macro'))
print("Politician Accuracy:", accuracy_score(y_pol, y_pol_pred))

### 2) Ablated Baseline Model


In [ ]:
import re
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import numpy as np
import torch
from sklearn.metrics import f1_score, accuracy_score
import matplotlib.pyplot as plt

In [ ]:
# 1. Extract top TF-IDF features from locked baseline
top_k = 25

# Top Post (label = 1)
top_positive_idx = np.argsort(coeffs)[-top_k:]
top_positive_words = [feature_names[i] for i in top_positive_idx]

# Top PRE (label = 0)
top_negative_idx = np.argsort(coeffs)[:top_k]
top_negative_words = [feature_names[i] for i in top_negative_idx]

# Combined important words
top_all_words = list(set(top_positive_words + top_negative_words))

print("Top POST words:", top_positive_words[:10])
print("Top PRE words:", top_negative_words[:10])

In [ ]:
# 2. Define removal function
def remove_words(text, words):
    text = str(text).lower()
    for word in words:
        text = re.sub(rf"\b{word}\b", "", text)
    return re.sub(r"\s+", " ", text).strip()

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

# 3. Apply ablations to ALL datasets (train, val, test)
for df_ in [train_df, val_df, test_df]:
    df_['ab_post'] = df_['clean_caption'].apply(lambda x: remove_words(x, top_positive_words))
    df_['ab_pre'] = df_['clean_caption'].apply(lambda x: remove_words(x, top_negative_words))
    df_['ab_all'] = df_['clean_caption'].apply(lambda x: remove_words(x, top_all_words))

In [ ]:
# 4. Define clean training function (fresh model each time)
def train_logreg(train_text, test_text, y_train, y_test, return_preds=False):
    vectorizer = TfidfVectorizer(max_features=5000)

    X_train = vectorizer.fit_transform(train_text)
    X_test = vectorizer.transform(test_text)

    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    if return_preds:
        return y_pred
    else:
        return f1_score(y_test, y_pred, average='macro')

In [ ]:
# 5. Run ablation experiments with validation set
ablation_baseline_f1 = train_logreg(train_df['clean_caption'], val_df['clean_caption'], y_train, y_val)

post_f1 = train_logreg(train_df['ab_post'], val_df['ab_post'], y_train, y_val)
pre_f1 = train_logreg(train_df['ab_pre'], val_df['ab_pre'], y_train, y_val)
all_f1 = train_logreg(train_df['ab_all'], val_df['ab_all'], y_train, y_val)

In [ ]:
# 6. Print results
print("\nAblation Results:")
print(f"\nAblation Baseline F1: {ablation_baseline_f1:.4f}")

print("\nRemove POST words:")
print(f"F1: {post_f1:.4f} | Drop: {ablation_baseline_f1 - post_f1:.4f}")

print("\nRemove PRE words:")
print(f"F1: {pre_f1:.4f} | Drop: {ablation_baseline_f1 - pre_f1:.4f}")

print("\nRemove ALL important words:")
print(f"F1: {all_f1:.4f} | Drop: {ablation_baseline_f1 - all_f1:.4f}")

In [ ]:
# Get predictions for ablated model
y_pred_ab_all = train_logreg(
    train_df['ab_all'],
    test_df['ab_all'],
    y_train,
    y_test,
    return_preds=True
)

# Confusion matrix for ablated model
cm_ab = confusion_matrix(y_test, y_pred_ab_all)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_ab, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title("Baseline Ablated Confusion Matrix")
plt.show()

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Ablation bar chart
labels = ['Baseline', 'No POST', 'No PRE', 'No ALL']
f1_scores = [ablation_baseline_f1, post_f1, pre_f1, all_f1]

plt.figure()
plt.bar(labels, f1_scores, color='#4C72B0')
plt.ylabel('Macro F1 Score')
plt.title('Baseline Ablation Study Results')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


In [ ]:
# Attention heatmap BERT attention
def plot_word_importance(text, vectorizer, model, title):

    # Transform text
    X = vectorizer.transform([text])

    # Get feature names
    feature_names = vectorizer.get_feature_names_out()

    # Get coefficients
    coeffs = model.coef_[0]

    # Get indices of words present
    indices = X.nonzero()[1]

    words = [feature_names[i] for i in indices]
    scores = [coeffs[i] for i in indices]

    if len(words) == 0:
        print("No known words in this sentence")
        return

    # Reshape for heatmap
    scores_array = np.array(scores).reshape(1, -1)

    plt.figure(figsize=(max(6, len(words)), 2))
    sns.heatmap(scores_array, annot=[words], fmt="", cbar=True)

    plt.title(title)
    plt.yticks([])
    plt.xticks([])
    plt.tight_layout()
    plt.show()

### 3) Fine-tuned BERT classifier

In [ ]:
torch.manual_seed(42)

# Convert pandas dataset to HuggingFace Dataset
train_dataset = Dataset.from_pandas(train_df[['clean_caption', 'label']])
val_dataset = Dataset.from_pandas(val_df[['clean_caption', 'label']])
test_dataset = Dataset.from_pandas(test_df[['clean_caption', 'label']])

In [ ]:
# BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize(example):
  return tokenizer(example['clean_caption'],
                   truncation=True,
                   padding='max_length',
                   max_length=150)

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

In [ ]:
# Format for PyTorch
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
val_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

# Load BERT model
bert_model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2, attn_implementation="eager")

In [ ]:
# Defining the computation of the metrics
def compute_metrics(eval_pred):
  logits, labels = eval_pred
  preds = np.argmax(logits, axis=-1)
  return{
      'f1': f1_score(labels, preds, average='macro'),
      "accuracy": accuracy_score(labels, preds)
  }

In [ ]:
# Training setup
from transformers import TrainingArguments
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

lr = 2e-5

training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=lr,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,

    logging_strategy="steps",
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=50,

    save_strategy="no"
)

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()
trainer.save_model("./bert_model")
tokenizer.save_pretrained("./bert_model")

eval_results = trainer.evaluate()
print("Validation F1:", eval_results['eval_f1'])

### 4) Ablated BERT

In [ ]:
# Create ablation text
train_df['bert_ab'] = train_df['clean_caption'].apply(lambda x: remove_words(x, top_all_words))
val_df['bert_ab'] = val_df['clean_caption'].apply(lambda x: remove_words(x, top_all_words))
test_df['bert_ab'] = test_df['clean_caption'].apply(lambda x: remove_words(x, top_all_words))

# Convert to HF Datasets
train_ab_dataset = Dataset.from_pandas(train_df[['bert_ab', 'label']])
val_ab_dataset = Dataset.from_pandas(val_df[['bert_ab', 'label']])
test_ab_dataset = Dataset.from_pandas(test_df[['bert_ab', 'label']])

# Tokenize
def tokenize_ab(example):
    return tokenizer(example['bert_ab'],
                     truncation=True,
                     padding='max_length',
                     max_length=150)

train_ab_dataset = train_ab_dataset.map(tokenize_ab, batched=True)
val_ab_dataset = val_ab_dataset.map(tokenize_ab, batched=True)
test_ab_dataset = test_ab_dataset.map(tokenize_ab, batched=True)

train_ab_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
val_ab_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
test_ab_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

In [ ]:
# Train BERT on Ablated Data
ab_model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased', num_labels=2, attn_implementation="eager")

ab_training_args = TrainingArguments(
    output_dir='./bert_ablation',
    learning_rate=lr,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,

    logging_strategy="steps",
    logging_steps=50,

    eval_strategy="steps",
    eval_steps=50,

    save_strategy="no"
)

ab_trainer = Trainer(
    model=ab_model,
    args=ab_training_args,
    train_dataset=train_ab_dataset,
    eval_dataset=val_ab_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

ab_trainer.train()
ab_trainer.save_model("./bert_ab_model")
tokenizer.save_pretrained("./bert_ab_model")

In [ ]:
# Evaluate and Compare

# Baseline BERT
predictions = trainer.predict(test_dataset)

y_base_pred = np.argmax(predictions.predictions, axis=1)
y_base_true = predictions.label_ids

bert_base_f1 = f1_score(y_base_true, y_base_pred, average='macro')


# Ablated BERT
ab_predictions = ab_trainer.predict(test_ab_dataset)

y_ab_pred = np.argmax(ab_predictions.predictions, axis=1)
y_true = ab_predictions.label_ids

bert_ab_f1 = f1_score(y_true, y_ab_pred, average='macro')

print("\nBERT ABLATION RESULTS:")
print(f"BERT Baseline F1: {bert_base_f1:.4f}")
print(f"BERT Ablated F1: {bert_ab_f1:.4f}")
print(f"Performance Drop (Increase if -): {bert_base_f1 - bert_ab_f1:.4f}")

In [ ]:
# Plot the loss curve
def extract_train_losses(logs):
    losses = []
    steps = []

    for log in logs:
        if "loss" in log:
            losses.append(log["loss"])
            steps.append(log["step"])

    return steps, losses

def extract_eval_losses(logs):
    losses = []
    steps = []
    for log in logs:
        if "eval_loss" in log:
            losses.append(log["eval_loss"])
            steps.append(log["step"])
    return steps, losses


# Original BERT
train_steps, train_losses = extract_train_losses(trainer.state.log_history)
eval_steps, eval_losses = extract_eval_losses(trainer.state.log_history)

# Ablated BERT
ab_train_steps, ab_train_losses = extract_train_losses(ab_trainer.state.log_history)
ab_eval_steps, ab_eval_losses = extract_eval_losses(ab_trainer.state.log_history)

# Plot all 4 lines
plt.figure()

# Original
plt.plot(train_steps, train_losses, label="Train (Original)", color='#4C72B0')
plt.plot(eval_steps, eval_losses, label="Val (Original)", color='#DD8452')

# Ablated
plt.plot(ab_train_steps, ab_train_losses, linestyle='--', label="Train (Ablated)", color='#4C72B0')
plt.plot(ab_eval_steps, ab_eval_losses, linestyle='--', label="Val (Ablated)", color='#DD8452')

plt.xlabel("Training Step")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss (Original vs. Ablated BERT)")

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Attention Heatmap Figure

# Pick example text
example_text = test_df[test_df['label']==1]['clean_caption'].iloc[0]

def plot_attention(bert_model, text, title):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=30
    )

    inputs = {k: v.to(bert_model.device) for k, v in inputs.items()}

    with torch.no_grad():
        # Force attention output
        outputs = bert_model(**inputs, output_attentions=True, return_dict=False)

    attentions = outputs[-1]

    if attentions is None:
        print("No attention returned")
        return

    # Last layer, first head
    attention = attentions[-1][0][0]

    # Tokens
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

    # Clean tokens
    cleaned_tokens = []
    indices = []

    for i, token in enumerate(tokens):
        if token not in ['[CLS]', '[SEP]'] and not token.startswith('##') and token != '#':
            cleaned_tokens.append(token)
            indices.append(i)

    # Filter attention matrix
    attention = attention[indices][:, indices]

    # Plot
    plt.figure(figsize=(6, 5))
    sns.heatmap(
        attention.cpu().numpy(),
        xticklabels=cleaned_tokens,
        yticklabels=cleaned_tokens,
        cmap="magma"
    )

    plt.title(title)
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

# Run both
plot_attention(bert_model, example_text, "Original BERT Attention Heatmap")
plot_attention(ab_model, example_text, "Ablated BERT Attention Heatmap")

In [ ]:
def get_preds(bert_model, dataset, batch_size=8):
    bert_model.eval()
    loader = DataLoader(dataset, batch_size=batch_size)

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(bert_model.device) for k, v in batch.items()}

            outputs = bert_model(
                input_ids=batch['input_ids'],
                attention_mask=batch['attention_mask']
            )

            logits = outputs.logits
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch['label'].cpu().numpy())

    return np.array(all_preds), np.array(all_labels)

In [ ]:
def tokenize_fixed(example):
    return tokenizer(
        example['clean_caption'],
        truncation=True,
        padding='max_length',
        max_length=150
    )

test_dataset_fixed = Dataset.from_pandas(test_df[['clean_caption', 'label']])
test_dataset_fixed = test_dataset_fixed.map(tokenize_fixed, batched=True)

test_dataset_fixed.set_format(
    type='torch',
    columns=['input_ids', 'attention_mask', 'label']
)

test_ab_dataset_fixed = Dataset.from_pandas(test_df[['bert_ab', 'label']])
test_ab_dataset_fixed = test_ab_dataset_fixed.map(tokenize_ab, batched=True)

test_ab_dataset_fixed.set_format(
    type='torch',
    columns=['input_ids', 'attention_mask', 'label']
)

In [ ]:
# OG BERT Results
y_pred, y_true = get_preds(bert_model, test_dataset_fixed)

print("\nOriginal BERT:")
print(classification_report(y_true, y_pred))
print("Macro F1:", f1_score(y_true, y_pred, average='macro'))

# Ablated BERT Results
y_ab_pred, y_ab_true = get_preds(ab_model, test_ab_dataset_fixed)

print("\nAblated BERT:")
print(classification_report(y_ab_true, y_ab_pred))
print("Macro F1:", f1_score(y_ab_true, y_ab_pred, average='macro'))

In [ ]:
# OG BERT group comparison
test_df = test_df.reset_index(drop=True)
test_df['pred'] = y_pred

inf = test_df[test_df['account_type'] == 'influencer']
pol = test_df[test_df['account_type'] == 'politician']

from sklearn.metrics import f1_score, accuracy_score
print("\nOriginal BERT Group Results:")

print("Influencer F1:", f1_score(inf['label'], inf['pred'], average='macro'))
print("Influencer Accuracy:", accuracy_score(inf['label'], inf['pred']))

print("Politician F1:", f1_score(pol['label'], pol['pred'], average='macro'))
print("Politician Accuracy:", accuracy_score(pol['label'], pol['pred']))

# Ablated BERT group comparison
test_df['pred_ablation'] = y_ab_pred

inf_ab = test_df[test_df['account_type'] == 'influencer']
pol_ab = test_df[test_df['account_type'] == 'politician']

print("\nAblated BERT Group Results:")

print("Influencer F1:", f1_score(inf_ab['label'], inf_ab['pred_ablation'], average='macro'))
print("Influencer Accuracy:", accuracy_score(inf_ab['label'], inf_ab['pred_ablation']))

print("Politician F1:", f1_score(pol_ab['label'], pol_ab['pred_ablation'], average='macro'))
print("Politician Accuracy:", accuracy_score(pol_ab['label'], pol_ab['pred_ablation']))

In [ ]:
# Confusion matrix for original BERT
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['PRE', 'POST'],
    yticklabels=['PRE', 'POST']
)

plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Original BERT Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix for ablated BERT
cm_ab = confusion_matrix(y_ab_true, y_ab_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm_ab,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['PRE', 'POST'],
    yticklabels=['PRE', 'POST']
)

plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Ablated BERT Confusion Matrix')
plt.tight_layout()
plt.show()


In [ ]:
# Compare baseline vs ablated
groups = ['Influencer', 'Politician']

bert_scores = [
    f1_score(inf['label'], inf['pred'], average='macro'),
    f1_score(pol['label'], pol['pred'], average='macro')
]

ab_scores = [
    f1_score(inf_ab['label'], inf_ab['pred_ablation'], average='macro'),
    f1_score(pol_ab['label'], pol_ab['pred_ablation'], average='macro')
]

x = np.arange(len(groups))
width = 0.35

plt.figure()
plt.bar(x - width/2, bert_scores, width, label='Original',color='#4C72B0')
plt.bar(x + width/2, ab_scores, width, label='Ablated',color='#DD8452')

plt.xticks(x, groups)
plt.ylabel('Macro F1 Score')
plt.title('BERT Performance by Group (Original vs. Ablated)')
plt.legend()
plt.ylim(0, 1)

plt.show()